In [ ]:
# ============================================================
# Notebook 10 — Global TFT Benchmark
# Cell 1 — Environment setup
# ============================================================

from pathlib import Path
import warnings
import random
import time

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

DATA_FOLDER = Path("../data/favorita")
CLEAN_DATA_PATH = DATA_FOLDER / "benchmark_300_cleaned.csv"

# ------------------------------------------------------------
# Forecast settings
# ------------------------------------------------------------

GROUP_COLUMNS = ["store_nbr", "item_nbr"]

MAX_ENCODER_LENGTH = 120
MAX_PREDICTION_LENGTH = 60
FORECAST_HORIZONS = [7, 14, 30, 60]

print("Notebook 10 environment ready.")
print("Clean dataset:", CLEAN_DATA_PATH)
print("Encoder length:", MAX_ENCODER_LENGTH)
print("Prediction length:", MAX_PREDICTION_LENGTH)

In [ ]:
# ============================================================
# Cell 2 — Load cleaned data and build TFT identifiers
# ============================================================

data = pd.read_csv(
    CLEAN_DATA_PATH,
    parse_dates=["date"]
)

# Sort each store-item series chronologically.
data = (
    data
    .sort_values(["store_nbr", "item_nbr", "date"])
    .reset_index(drop=True)
)

# Unique identifier for each time series.
data["series_id"] = (
    data["store_nbr"].astype(str)
    + "_"
    + data["item_nbr"].astype(str)
)

# IMPORTANT:
# time_idx must progress independently within each series.
data["time_idx"] = (
    data.groupby("series_id")
    .cumcount()
    .astype(int)
)

print("Shape:", data.shape)
print("Series:", data["series_id"].nunique())
print("Date range:", data["date"].min(), "to", data["date"].max())
print("Missing demand targets:", data["demand_target"].isna().sum())

# Sanity check: each series should begin at time_idx = 0.
time_idx_check = (
    data.groupby("series_id")["time_idx"]
    .agg(["min", "max", "count"])
)

print("\nAll series start at 0:",
      (time_idx_check["min"] == 0).all())

print("Example:")
print(
    data[
        ["series_id", "date", "time_idx", "demand_target"]
    ].head(10)
)

In [ ]:
# ============================================================
# Cell 3 — Audit unknown targets before TFT dataset creation
# ============================================================

unknown_targets = data[
    data["demand_target"].isna()
].copy()

print("Unknown target rows:", len(unknown_targets))

print("\nUnknown rows by store_operating:")
print(
    unknown_targets["store_operating"]
    .value_counts(dropna=False)
)

print("\nUnknown rows by was_observed:")
print(
    unknown_targets["was_observed"]
    .value_counts(dropna=False)
)

print("\nExample unknown rows:")
print(
    unknown_targets[
        [
            "series_id",
            "date",
            "time_idx",
            "store_operating",
            "was_observed",
            "demand_target"
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# Cell 4 — Safe target handling for TFT
# ============================================================

# 1 = real demand target
# 0 = target unavailable / should not contribute to training loss
data["target_known"] = (
    data["demand_target"].notna()
).astype(int)

# TFT cannot work directly with NaN targets.
# We therefore create a separate modeling target.
#
# IMPORTANT:
# These zeros are placeholders only.
# Unknown rows receive zero training weight below.
data["tft_target"] = (
    data["demand_target"]
    .fillna(0.0)
    .astype(float)
)

# Zero-weight unknown targets so they do not contribute
# to the model's training loss.
data["target_weight"] = (
    data["target_known"]
    .astype(float)
)

print("Original unknown targets:",
      data["demand_target"].isna().sum())

print("NaNs in TFT target:",
      data["tft_target"].isna().sum())

print("Zero-weight rows:",
      (data["target_weight"] == 0).sum())

print("Positive-weight rows:",
      (data["target_weight"] == 1).sum())

In [ ]:
# ============================================================
# Cell 5 — Check TFT / PyTorch Forecasting libraries
# ============================================================

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

try:
    import lightning.pytorch as pl
    print("Lightning available.")
except ImportError:
    print("Lightning NOT installed.")

try:
    import pytorch_forecasting
    from pytorch_forecasting import TimeSeriesDataSet
    from pytorch_forecasting.models import TemporalFusionTransformer

    print("PyTorch Forecasting version:", pytorch_forecasting.__version__)
    print("PyTorch Forecasting available.")
except ImportError:
    print("PyTorch Forecasting NOT installed.")

In [ ]:
# ============================================================
# Cell 6 — Prepare TFT dataset columns
# ============================================================

# Static categorical features: constant for each series
STATIC_CATEGORICALS = [
    "series_id",
    "store_nbr",
    "item_nbr",
]

# Known future categorical features
TIME_VARYING_KNOWN_CATEGORICALS = [
    "promotion_status",
]

# Known future real-valued features
TIME_VARYING_KNOWN_REALS = [
    "time_idx",
    "target_known",
]

# Historical observed real-valued features
TIME_VARYING_UNKNOWN_REALS = [
    "tft_target",
]

# Convert categorical columns to strings
# because PyTorch Forecasting expects categorical values cleanly encoded.
for col in STATIC_CATEGORICALS + TIME_VARYING_KNOWN_CATEGORICALS:
    data[col] = data[col].astype(str)

print("Static categoricals:", STATIC_CATEGORICALS)
print("Known categoricals:", TIME_VARYING_KNOWN_CATEGORICALS)
print("Known reals:", TIME_VARYING_KNOWN_REALS)
print("Unknown reals:", TIME_VARYING_UNKNOWN_REALS)

print("\nRows:", len(data))
print("Series:", data["series_id"].nunique())

In [ ]:
# ============================================================
# Cell 6 — TFT framework setup
# ============================================================

import torch
import lightning.pytorch as pl

from lightning.pytorch.callbacks import (
    EarlyStopping,
    LearningRateMonitor
)

from pytorch_forecasting import (
    TimeSeriesDataSet,
    TemporalFusionTransformer
)

from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

pl.seed_everything(SEED, workers=True)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Training device:", DEVICE)


# ------------------------------------------------------------
# TFT forecast loss
# ------------------------------------------------------------
#
# Instead of predicting only one number, TFT can predict
# several demand quantiles.
#
# P50 = median / central forecast
# P90 = demand level we expect to stay below ~90% of the time
#
# These will later be useful for inventory risk and safety stock.
# ------------------------------------------------------------

TFT_QUANTILES = [0.1, 0.5, 0.9]

TFT_LOSS = QuantileLoss(
    quantiles=TFT_QUANTILES
)


# ------------------------------------------------------------
# Training callbacks
# ------------------------------------------------------------

early_stop_callback = EarlyStopping(
    monitor="val_loss",
    min_delta=1e-4,
    patience=3,
    verbose=True,
    mode="min"
)

learning_rate_monitor = LearningRateMonitor(
    logging_interval="epoch"
)


print("TFT libraries ready.")
print("Quantiles:", TFT_QUANTILES)
print("Early stopping patience:", early_stop_callback.patience)

In [ ]:
# ============================================================
# Cell 7 — Define TFT feature groups
# ============================================================

# ------------------------------------------------------------
# Static categorical features
# These do not change over time for a given series.
# ------------------------------------------------------------

STATIC_CATEGORICALS = [
    "series_id",
    "store_nbr",
    "item_nbr",
]

# ------------------------------------------------------------
# Known future categorical features
# We know these values in advance for future dates.
# ------------------------------------------------------------

TIME_VARYING_KNOWN_CATEGORICALS = [
    "promotion_status",
]

# ------------------------------------------------------------
# Known future real-valued features
# time_idx is always known.
# target_known tells us whether the original target was observed.
# ------------------------------------------------------------

TIME_VARYING_KNOWN_REALS = [
    "time_idx",
    "target_known",
]

# ------------------------------------------------------------
# Historical / unknown real-valued features
# We only know demand once it actually happens.
# ------------------------------------------------------------

TIME_VARYING_UNKNOWN_REALS = [
    "tft_target",
]

# ------------------------------------------------------------
# Convert categorical variables to strings.
# PyTorch Forecasting will encode these internally.
# ------------------------------------------------------------

categorical_columns = (
    STATIC_CATEGORICALS
    + TIME_VARYING_KNOWN_CATEGORICALS
)

for column in categorical_columns:
    data[column] = data[column].astype(str)

print("Static categoricals:")
print(STATIC_CATEGORICALS)

print("\nKnown future categoricals:")
print(TIME_VARYING_KNOWN_CATEGORICALS)

print("\nKnown future reals:")
print(TIME_VARYING_KNOWN_REALS)

print("\nUnknown historical reals:")
print(TIME_VARYING_UNKNOWN_REALS)

print("\nRows:", len(data))
print("Series:", data["series_id"].nunique())

In [ ]:
# ============================================================
# Cell 8 — Add TFT calendar and static metadata features
# ============================================================

# Calendar features known in advance
data["day_of_week"] = data["date"].dt.dayofweek.astype(str)
data["month"] = data["date"].dt.month.astype(str)
data["day_of_month"] = data["date"].dt.day.astype(float)

# Load static item/store information
items = pd.read_csv(DATA_FOLDER / "items.csv")
stores = pd.read_csv(DATA_FOLDER / "stores.csv")

items = items[
    ["item_nbr", "family", "class", "perishable"]
].copy()

stores = stores[
    ["store_nbr", "type", "cluster"]
].rename(columns={"type": "store_type"})

# Convert merge keys back temporarily because Cell 7 made them strings
data["store_nbr"] = data["store_nbr"].astype(int)
data["item_nbr"] = data["item_nbr"].astype(int)

data = (
    data
    .merge(items, on="item_nbr", how="left")
    .merge(stores, on="store_nbr", how="left")
)

# TFT categorical values should be strings
categorical_cols = [
    "series_id",
    "store_nbr",
    "item_nbr",
    "family",
    "class",
    "store_type",
    "cluster",
    "promotion_status",
    "day_of_week",
    "month",
]

for col in categorical_cols:
    data[col] = data[col].astype(str)

print("Rows:", len(data))
print("Series:", data["series_id"].nunique())
print("Calendar + metadata ready.")

In [ ]:
# ============================================================
# Cell 9 — CORRECTED TFT feature configuration
# ============================================================

STATIC_CATEGORICALS = [
    "series_id",
    "store_nbr",
    "item_nbr",
    "family",
    "class",
    "store_type",
    "cluster",
]

TIME_VARYING_KNOWN_CATEGORICALS = [
    "promotion_status",
    "day_of_week",
    "month",
]

# Only information genuinely available for future dates.
TIME_VARYING_KNOWN_REALS = [
    "time_idx",
    "day_of_month",
    "is_holiday",
    "is_event",
    "is_additional",
    "is_transfer",
    "is_bridge",
    "is_work_day",
]

# These are only observed historically.
TIME_VARYING_UNKNOWN_REALS = [
    "tft_target",
    "target_known",
]

data["time_idx"] = data["time_idx"].astype(int)

for col in [
    "day_of_month",
    "is_holiday",
    "is_event",
    "is_additional",
    "is_transfer",
    "is_bridge",
    "is_work_day",
    "target_known",
]:
    data[col] = data[col].astype(float)

print("Corrected TFT feature configuration ready.")
print("No target-derived variable is treated as future-known.")

In [25]:
# ============================================================
# Cell 10 — Build TFT training and validation datasets
# ============================================================

TRAIN_END = pd.Timestamp("2017-04-30")
VALIDATION_START = pd.Timestamp("2017-05-01")
VALIDATION_END = pd.Timestamp("2017-06-29")

# Training contains no future information.
training_data = data[
    data["date"] <= TRAIN_END
].copy()

# Dataset used to construct the validation encoder + 60-day decoder.
validation_data = data[
    data["date"] <= VALIDATION_END
].copy()

training_dataset = TimeSeriesDataSet(
    training_data,

    time_idx="time_idx",
    target="tft_target",
    weight="target_weight",

    group_ids=["series_id"],

    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,

    static_categoricals=STATIC_CATEGORICALS,

    time_varying_known_categoricals=
        TIME_VARYING_KNOWN_CATEGORICALS,

    time_varying_known_reals=
        TIME_VARYING_KNOWN_REALS,

    time_varying_unknown_reals=
        TIME_VARYING_UNKNOWN_REALS,

    target_normalizer=GroupNormalizer(
        groups=["series_id"],
        transformation="softplus"
    ),

    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,

    allow_missing_timesteps=False,
)

# Same encoders/scalers as training.
# predict=True creates the final 60-day forecast window
# for every series.
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    validation_data,
    predict=True,
    stop_randomization=True,
)

print("Training sequences:", len(training_dataset))
print("Validation sequences:", len(validation_dataset))

Training sequences: 394530
Validation sequences: 300


In [26]:
# ============================================================
# Cell 11 — Create DataLoaders
# ============================================================

BATCH_SIZE = 64

train_dataloader = training_dataset.to_dataloader(
    train=True,
    batch_size=BATCH_SIZE,
    num_workers=0,   # safest setting on Windows
)

val_dataloader = validation_dataset.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

print("Training batches:", len(train_dataloader))
print("Validation batches:", len(val_dataloader))

Training batches: 6164
Validation batches: 5


In [27]:
# ============================================================
# Cell 12 — Create global Temporal Fusion Transformer
# ============================================================

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,

    learning_rate=0.001,

    # Small enough for CPU training, but still a real TFT.
    hidden_size=16,
    attention_head_size=4,
    hidden_continuous_size=8,

    dropout=0.1,

    # P10, P50 and P90
    output_size=len(TFT_QUANTILES),

    loss=TFT_LOSS,

    reduce_on_plateau_patience=2,

    log_interval=50,
)

total_parameters = sum(
    p.numel() for p in tft.parameters()
)

print("TFT created.")
print("Trainable parameters:", f"{total_parameters:,}")

TFT created.
Trainable parameters: 43,987


In [ ]:
# ============================================================
# Cell 13 — 1-EPOCH CPU SMOKE TEST
# ============================================================

smoke_early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    mode="min",
)

trainer = pl.Trainer(
    max_epochs=1,
    accelerator="cpu",
    devices=1,

    gradient_clip_val=0.1,

    callbacks=[
        smoke_early_stop,
    ],

    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
)

start_time = time.time()

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

SMOKE_TEST_MINUTES = (
    time.time() - start_time
) / 60

print("\n==============================")
print("TFT SMOKE TEST COMPLETE")
print("==============================")
print(
    "1 epoch runtime:",
    round(SMOKE_TEST_MINUTES, 2),
    "minutes"
)

In [28]:
# ============================================================
# Cell 14 — Increase batch size for CPU training
# ============================================================

FAST_BATCH_SIZE = 256

train_dataloader_fast = training_dataset.to_dataloader(
    train=True,
    batch_size=FAST_BATCH_SIZE,
    num_workers=0
)

val_dataloader_fast = validation_dataset.to_dataloader(
    train=False,
    batch_size=FAST_BATCH_SIZE,
    num_workers=0
)

print("Batch size:", FAST_BATCH_SIZE)
print("Training batches:", len(train_dataloader_fast))
print("Validation batches:", len(val_dataloader_fast))

Batch size: 256
Training batches: 1541
Validation batches: 2


In [ ]:
# ============================================================
# Cell 15 — 1-EPOCH FAST CPU TIMING TEST
# ============================================================

# Recreate a fresh TFT so this timing test starts clean.
tft_fast_test = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=0.001,
    hidden_size=16,
    attention_head_size=4,
    hidden_continuous_size=8,
    dropout=0.1,
    output_size=len(TFT_QUANTILES),
    loss=TFT_LOSS,
    reduce_on_plateau_patience=2,
    log_interval=50,
)

trainer_fast_test = pl.Trainer(
    max_epochs=1,
    accelerator="cpu",
    devices=1,
    gradient_clip_val=0.1,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
)

start_time = time.time()

trainer_fast_test.fit(
    tft_fast_test,
    train_dataloaders=train_dataloader_fast,
    val_dataloaders=val_dataloader_fast,
)

FAST_SMOKE_MINUTES = (
    time.time() - start_time
) / 60

print("\n==============================")
print("FAST TFT SMOKE TEST COMPLETE")
print("==============================")
print(
    "1 epoch runtime:",
    round(FAST_SMOKE_MINUTES, 2),
    "minutes"
)

In [29]:
# ============================================================
# Cell 16 — FINAL GLOBAL TFT TRAINING
# ============================================================

from lightning.pytorch.callbacks import ModelCheckpoint

# ------------------------------------------------------------
# Save the best model based on validation loss
# ------------------------------------------------------------

checkpoint_callback = ModelCheckpoint(
    dirpath="../models/tft",
    filename="global-tft-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss",
    mode="min",
    save_top_k=1,
)

# Stop if validation loss does not improve for 2 epochs
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    min_delta=1e-4,
    patience=2,
    verbose=True,
    mode="min",
)

# ------------------------------------------------------------
# Create a fresh TFT model
# ------------------------------------------------------------

tft_final = TemporalFusionTransformer.from_dataset(
    training_dataset,

    learning_rate=0.001,

    hidden_size=16,
    attention_head_size=4,
    hidden_continuous_size=8,

    dropout=0.1,

    # Predict P10, P50, P90
    output_size=len(TFT_QUANTILES),
    loss=TFT_LOSS,

    reduce_on_plateau_patience=2,

    # Reduce logging overhead
    log_interval=100,
)

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in tft_final.parameters()):,}"
)

# ------------------------------------------------------------
# Trainer
# ------------------------------------------------------------

trainer_final = pl.Trainer(
    max_epochs=8,

    accelerator="cpu",
    devices=1,

    gradient_clip_val=0.1,

    callbacks=[
        early_stop_callback,
        checkpoint_callback,
    ],

    logger=False,
    enable_model_summary=False,
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

start_time = time.time()

trainer_final.fit(
    tft_final,
    train_dataloaders=train_dataloader_fast,
    val_dataloaders=val_dataloader_fast,
)

TOTAL_TRAINING_MINUTES = (
    time.time() - start_time
) / 60

print("\n====================================")
print("FINAL TFT TRAINING COMPLETE")
print("====================================")

print(
    "Training time:",
    round(TOTAL_TRAINING_MINUTES, 2),
    "minutes"
)

print(
    "Best checkpoint:",
    checkpoint_callback.best_model_path
)

print(
    "Best validation loss:",
    float(checkpoint_callback.best_model_score)
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Trainable parameters: 43,987


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved. New best score: 2.078


... (7 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 2 records. Best score: 2.026. Signaling Trainer to stop.



FINAL TFT TRAINING COMPLETE
Training time: 191.97 minutes
Best checkpoint: C:\Users\karra\Desktop\inventory-forecasting\models\tft\global-tft-epoch=04-val_loss=2.0258.ckpt
Best validation loss: 2.0257842540740967


In [31]:
# ============================================================
# Cell 17 — SAVE ALL IMPORTANT TFT ARTIFACTS
# ============================================================

from pathlib import Path
import json
import pickle
import shutil

SAVE_FOLDER = Path("../models/tft")
SAVE_FOLDER.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Copy best trained TFT checkpoint to a permanent filename
# ------------------------------------------------------------

best_checkpoint_path = Path(
    checkpoint_callback.best_model_path
)

FINAL_MODEL_PATH = (
    SAVE_FOLDER / "final_global_tft.ckpt"
)

shutil.copy2(
    best_checkpoint_path,
    FINAL_MODEL_PATH
)

# ------------------------------------------------------------
# Save prepared TFT dataframe
# Pickle works without installing pyarrow/fastparquet
# ------------------------------------------------------------

PREPARED_DATA_PATH = (
    DATA_FOLDER / "tft_300_prepared.pkl"
)

data.to_pickle(
    PREPARED_DATA_PATH
)

# ------------------------------------------------------------
# 3. Save TimeSeriesDataSet configuration / encoders / scalers
# ------------------------------------------------------------

DATASET_PARAMETERS_PATH = (
    SAVE_FOLDER / "tft_dataset_parameters.pkl"
)

with open(DATASET_PARAMETERS_PATH, "wb") as f:
    pickle.dump(
        training_dataset.get_parameters(),
        f
    )

# ------------------------------------------------------------
# 4. Save experiment configuration
# ------------------------------------------------------------

experiment_config = {
    "seed": SEED,

    "max_encoder_length": MAX_ENCODER_LENGTH,
    "max_prediction_length": MAX_PREDICTION_LENGTH,

    "forecast_horizons": FORECAST_HORIZONS,

    "quantiles": TFT_QUANTILES,

    "batch_size": FAST_BATCH_SIZE,

    "learning_rate": 0.001,
    "hidden_size": 16,
    "attention_head_size": 4,
    "hidden_continuous_size": 8,
    "dropout": 0.1,

    "max_epochs": 8,
    "early_stopping_patience": 2,

    "train_end": str(TRAIN_END.date()),
    "validation_start": str(VALIDATION_START.date()),
    "validation_end": str(VALIDATION_END.date()),

    "static_categoricals":
        STATIC_CATEGORICALS,

    "time_varying_known_categoricals":
        TIME_VARYING_KNOWN_CATEGORICALS,

    "time_varying_known_reals":
        TIME_VARYING_KNOWN_REALS,

    "time_varying_unknown_reals":
        TIME_VARYING_UNKNOWN_REALS,

    "best_validation_loss":
        float(checkpoint_callback.best_model_score),

    "training_minutes":
        float(TOTAL_TRAINING_MINUTES),
}

CONFIG_PATH = (
    SAVE_FOLDER / "tft_experiment_config.json"
)

with open(CONFIG_PATH, "w") as f:
    json.dump(
        experiment_config,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 5. Confirmation
# ------------------------------------------------------------

print("====================================")
print("TFT ARTIFACTS SAVED")
print("====================================")

print("Final model:")
print(FINAL_MODEL_PATH)

print("\nPrepared data:")
print(PREPARED_DATA_PATH)

print("\nDataset parameters:")
print(DATASET_PARAMETERS_PATH)

print("\nExperiment configuration:")
print(CONFIG_PATH)

print("\nSAFE TO RESTART KERNEL.")

TFT ARTIFACTS SAVED
Final model:
..\models\tft\final_global_tft.ckpt

Prepared data:
..\data\favorita\tft_300_prepared.pkl

Dataset parameters:
..\models\tft\tft_dataset_parameters.pkl

Experiment configuration:
..\models\tft\tft_experiment_config.json

SAFE TO RESTART KERNEL.
